SUMMARIZATION AUTOMATICA: L'ARTE DELLA SINTESI NEURALE

La summarization automatica è il compito di far leggere a un modello un testo lungo e produrre una versione più corta che mantenga le informazioni importanti.

testo lungo -> modelli NPL / Transformer -> selezione e riorganizzazione delle informazioni -> riassunto

Ci sono due famiglie diverse di summarization: etrattiva e astrattiva.

** Summarization estrattiva **
sceglier la fedeltà letterale

La summarizatoin estrattiva, prende pezzi di testo originale e li seleziona.
Si individuano le frasi più pesanti di un testo e le si mettono insieme.
quindi:
testo originale -> scelgo le frasi più importanti -> le riutilizzo quasi identiche

Esempio
Testo:
"Il mercato è cresciuto del 10%.
L'azienda ha aperto una nuova sede.
I costi energetici sono aumentati.
Il management prevede nuovi investimenti."
Riassunto:
"Il mercato è cresciuto del 10%.
Il management prevede nuovi investimenti."

Non sto veramente riscrivendo

** Summarization astrattiva **
scegliere la rielaborazione

La Summarization astrattiva invece fa qualcosa di più simile a quello che farebbe una persona.
Legge, capisce e poi riscrive da zero, quindi:
testo lungo -> comprensione/rappresentazione del contenuto -> generazione di nuove frasi -> riasunto
Esempio partendo dalla frase precedente
Riassunto:
"L'azienda cresce e prevede nuovi investimenti,
nonostante l'aumento dei costi energetici."
Questa frase non esiste necessariamente nel testo originale, Il modello ha sintetizzato e riformulato.

Quiondi la distinzione fondamentale è:
ESTRATTIVA -> sceglie parti del testo
ASTRATTIVA -> genera un nuovo testo sintetico

Quando si parla di sintesi neurale, si solito ci si riferisce soprattutto alla summarization astrattiva moderna basata su Transformer.

Il problema diventa quindi un classico problema di sequence-to-sequence (Seq2Seq)
Seq2Seq l'abbiamo già vista per la traduzione, qui la applichiamo per il riassunto corto
Il task è differente, ma l'architettura concettuale è molto simile.

Testo lungo -> tekenizer -> token IDs -> encoder -> rappresentazione contestuale del testo -> decoder -> genera il riassunto token per token

L'encoder legge tutto il documento e costruisce rappresentazioni contestuali. Il decoder genera il riassunto un token alla volta, usanto sia ciò che ha già generato, sia le informazioni provenienti dall'encoder.

Il meccanismo di attention è fondamentale perchè il decoder non deve comprimere necessariamente tutto il documento in un singolo vettore fisso.
Può, ad ogni passo, 'guardare' le parti dell'input più rilevanti e dare attenzione alla parte maggiormente rappresentativa
Quindi: attention -> quali parti del documento sono utili per generare questo token del riassunto?

Il modello viene normalmente addestrato per coppie:
documento -> riassunto corretto
Durante il training:
documento -> modello -> riassunto previsto -> confronto con riassunto corretto -> loss -> backpropagation -> aggiornamento pesi
Quindi, lo stesso principio visto con GPT, la differenza è il task
Nel GPT autoregressivo - input: predici il prossimo token
Nella summarization encoder-encoder - documento, genera sequenza riassuntiva

Meccanismi di Selezione e Generazione
- Ranking delle frasi: nell'estrattiva si calcola un punteggio di importanza basato sulla centralità semantica di ogni periodo
- Paratesto e Parafrasi: l'astrattiva richiede capacità di Natural Language Generation (NLG) per mantenere la coerenza stilistica
- Fidelity vs Fluency: l'estrattiva è sempre fedele ai fatti ma meno fluida; l'astrattiva è fluida ma rischia allucinazione
- Compressione: rapporto tra il numero di token del testo originale e quelli del riassunto finale

Modelli classici usati normalmente per la summarization sono: BART e T5. Non sono BERT puro e non sono GPT-2 puro: sono modelli progettati in modo diverso, tipicamente encoder-encoder, molto adatti a task text-to-text come la traduzione, summarization e trasformazione del testo.
Mentre BERT è focalizzato sulla compressione (Encoder), modelli come BART e T5 utilizzano l'intera struttura Encoder-Decoder per gestire task generativi

Innovazione Architetturali
Specializzazione per la generazione
- BART (Bidirectional Auto-Regressive): addestrato corrompendo il testo con maschere e forzando il modello a ricostruire l'originale
- T5 (Text-to-text Transfer Transformer): ogni problema di NLP viene trattato come una conversioe da testo a testo.
- Encoder-Decoder Attention: il decoder guarda costantemente le rappresentazioni ricche dell'encoder per guidare la generazione, sull'intero documento originale.
- Denoising Objective: l'apprendimento basato sulla correzione di testo distorto prepara il modello a sintetizzare testi complessi.

L'Obbiettivo di TD
Dall'input al task prefix
Il modello T5 utilizza un prefisso testuale per identificare il compito. In questo caso, aggiungere una stringa 'summarize:' all'input attiva i pesi ottimizzati per la sintesi.
Questo approccio unificato permette di condividere la conoscenze semantica tra più task linguistici diversi. Lo stesso modelli, con suffissi diversi, può tradurre, correggere o riassumere, con la stessa profonda conoscenza della lingua

La summarization può anche essere fatta oggi con LLM decoder-only, come i modelli GPT moderni, semplicemente tramite il prompt:
"rissumi questo testo in 5 righe: ... testo ..."
In questo caso, non stai utilizzando un modello specificatamente costruito come summarizer, stai usanto la capacità generativa generale dell'LLM

Un concetto importante è la compression ratio
Supponiamo
testo originale: 1000 parole
riassunto: 100 parole
hai una compressoine forte
Il modello deve quindi bilanciare brevità vs completezz
Troppo corto -> perdi informazioni importanti
troppo lungo -> non è più realmente un riassunto

Ci sono poi 3 qualità fondamentali di un buon summarizer:
1) rilevanza: deve mantenere le informazioni importanti
2) coerenza: il riassunto deve avere senso come testo
3) fedeltà: non deve inventare informazioni.
Questo ultimo punto è particolarmente importante in un summarization astrattiva
Perhè essendo generativa puà accadere:
testo: "la vendite sono aumentate del 7%"
modello: "le vendite sono aumentate del 12%"
il testo è grammaticalmente perfetto, ma il riassunto sbagliato.
Questa è una forma di allucinazione (hallucination)

C'è poi il problema della lunghezza massima
Un Transformer non può necessariamente ricevere un documento di lunghezza arbitraria
Seil modello accetta, per esempio, una certa quantità massima di token e tu gli dai un documento molto più lungo, devi gestire la situazione.
Possibili strategie:
- dividere per chunk: documento enorme -> riassunto chunk 1 + riassunto chunk 2 + riassunto chunk 3 + riassunto chunk ... -> n mini-riassunti -> risultato finale
Ma bisogna fare attenzione, perchè più livelli (piu chunk) di riassunto possono introdurre perdita di dettagli.

Per valutare un sistema sdi summarization incontri spesso ROUGE

ROUGE confronta il riassunto generato con uno o più riassunti di riferimento
Valutare un riassunto è soggettivo perchè non esiste una singola risposta corretta. Abbiamo bisogno di metriche che confrontino il modello con riferimenti umani.
Con Rouge più parole ci sono corrisondenti tra il riassunto automatico ed il riassunto umano, più il punteggio è alto.
Ma Rouge non misura perfettamente la qualità semantica. Due riassunti possono significare la stessa cosa ma usare parole differenti.
Per questo oggi si usano anche metriche semantiche, valutazione umana e metrihe basate su modelli.

Varianti di Rouge
Metriche di overlap
- Rouge-N: misura la sovrapposizione di sequenze di n parole (es. Rouge-1 per parole singole, Rouge-2 per coppie, ecc)
- Rouge-L: basato sulla Longest Common Subsequences (LCS), valuta la struttura della frase senza richiedere n-grammi consecutivi. 
- Recall vs Precision: la recall misura quanto del riassunto umano è stato catturato; la precision quanto del riassunto generato è rilevante
- Rouge-W: una variante pesato che valorizza maggiormente le sequenze di parole consecutive corrette.

Limiti delle Metriche automatiche
- Mancanza di semantica: Rouge penalizza i sinonimi: se il modello usa 'veloce' e l'umano 'rapido', il puntegggio diminuisce nonostante il senso sia identico
- Focus sulla Quantità: un punteggio Rouge alto non garantisce che il testo sia fattualmente corretto; un'allucinazione che contiene molte parole dell'originale può aver un buon punteggio.
- Humman Evaluation: resta essenziale integrare le metriche automatiche con valutazioni umane su criteri di informatività, concisione, e assenza di errori.


In [1]:
# python -m pip install rouge-score
"""
Confronto Summarization: BART vs T5 con Keras 3 e PyTorch Backend (Best Practices 2026)
---------------------------------------------------------------------------------------
In questo script confronteremo due dei pesi massimi dell'NLP per il riassunto astrattivo.
Utilizzeremo l'ecosistema Keras 3 con backend PyTorch per massimizzare la flessibilità 
e le performance su GPU.

Modelli:
1. BART (Facebook/Meta): Architettura encoder-decoder pre-addestrata come denoising autoencoder.
2. T5 (Google): Approccio text-to-text (ogni task è visto come una conversazione stringa-a-stringa).

Valutazione:
Useremo la metrica ROUGE (Recall-Oriented Understudy for Gisting Evaluation) per misurare
la sovrapposizione tra i riassunti generati e la ground truth.
"""

import os

# 1. SETUP BACKEND (Best Practice 2026: Definire il backend prima di importare Keras)
os.environ["KERAS_BACKEND"] = "torch"

import keras
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import pandas as pd
from rouge_score import rouge_scorer

def summarize_text(text, model_name, device, max_length=150):
    """
    Funzione universale per la summarization utilizzando l'ecosistema Transformers
    integrato con il backend Keras/PyTorch.
    """
    print(f"\n--- Generazione con {model_name} ---")
    
    # Caricamento Tokenizer e Modello
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    model.to(device)
    
    # Best Practice: Gestione specifica per T5 (richiede il prefisso del task)
    if "t5" in model_name.lower():
        text = "summarize: " + text

    # Tokenizzazione (Padding e Truncation automatici)
    inputs = tokenizer(
        text, 
        return_tensors="pt", 
        max_length=1024, 
        truncation=True
    ).to(device)

    # Generazione con Beam Search (standard per summarization di qualità)
    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=max_length,
        min_length=40,
        length_penalty=2.0,
        num_beams=4,
        early_stopping=True
    )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

def evaluate_rouge(generated_summary, reference_summary):
    """
    Calcola i punteggi ROUGE-1, ROUGE-2 e ROUGE-L.
    """
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = scorer.score(reference_summary, generated_summary)
    
    # Formattiamo i risultati per una lettura pulita
    formatted_scores = {key: round(value.fmeasure * 100, 2) for key, value in scores.items()}
    return formatted_scores

def main():
    # Setup Hardware
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Esecuzione su: {device.upper()}")

    # --- DATI DI ESEMPIO (Test di realtà) ---
    original_text = """
    L'intelligenza artificiale generativa ha subito un'accelerazione senza precedenti nel 2025. 
    L'integrazione di architetture multimodali ha permesso ai modelli non solo di scrivere testi, 
    ma di comprendere video e audio in tempo reale con una precisione sovrumana. 
    Keras 3 è diventato lo standard industriale grazie alla sua capacità di astrarre il framework 
    sottostante, permettendo agli sviluppatori di passare da PyTorch a JAX o TensorFlow con una 
    sola riga di codice. Molti esperti ritengono che questa flessibilità sia la chiave per 
    ridurre i costi di addestramento e migliorare l'efficienza energetica dei data center.
    """
    
    reference_summary = "L'AI generativa nel 2025 è diventata multimodale e precisa. Keras 3 è lo standard grazie alla sua flessibilità tra backend come PyTorch, ottimizzando costi ed energia."

    # --- MODELLI DA CONFRONTARE ---
    models_to_test = {
        "BART": "facebook/bart-large-cnn", # Ottimo per riassunti strutturati
        "T5": "t5-base"                    # Versatile e bilanciato
    }

    results = []

    for name, model_id in models_to_test.items():
        try:
            # Generazione
            summary = summarize_text(original_text, model_id, device)
            
            # Valutazione
            scores = evaluate_rouge(summary, reference_summary)
            
            # Archiviazione
            results.append({
                "Modello": name,
                "Riassunto": summary,
                **scores
            })
            
            print(f"Riassunto: {summary}")
            print(f"Scores: {scores}")
            
        except Exception as e:
            print(f"Errore durante il test di {name}: {e}")

    # --- REPORT FINALE ---
    print("\n" + "="*50)
    print("CONFRONTO FINALE PRESTAZIONI")
    print("="*50)
    df = pd.DataFrame(results)
    # Rimuoviamo il testo del riassunto dalla tabella per chiarezza nel print
    print(df.drop(columns=["Riassunto"]).to_string(index=False))

if __name__ == "__main__":
    main()

c:\EPICODE\Epicode_Python_AI_MachineLearning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Esecuzione su: CPU

--- Generazione con facebook/bart-large-cnn ---


c:\EPICODE\Epicode_Python_AI_MachineLearning\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\barbara\.cache\huggingface\hub\models--facebook--bart-large-cnn. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 
Loa

Riassunto: L'intelligenza artificiale generativa ha subito un'accelerazione senza precedenti nel 2025. L'integrazione di architetture multimodali ha permesso ai modelli non solo di scrivere testi, but also di comprendere video e audio in tempo reale.
Scores: {'rouge1': 19.67, 'rouge2': 3.39, 'rougeL': 16.39}

--- Generazione con t5-base ---


c:\EPICODE\Epicode_Python_AI_MachineLearning\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\barbara\.cache\huggingface\hub\models--t5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 257/257 [00:00<00:00, 3549.05it/s]


Riassunto: Keras 3 è diventato lo standard industriale grazie alla sua capacità di astrarre il framework sottostante . agli sviluppatori di passare da PyTorch a JAX o TensorFlow con una sola riga di codice.
Scores: {'rouge1': 28.57, 'rouge2': 14.81, 'rougeL': 28.57}

CONFRONTO FINALE PRESTAZIONI
Modello  rouge1  rouge2  rougeL
   BART   19.67    3.39   16.39
     T5   28.57   14.81   28.57
